# 6. McNemar y bootstrap pareado, con el umbral elegido solo con datos de entrenamiento

DeLong (sección 5) compara la capacidad de ordenar casos de dos modelos, es decir el AUC, con una fórmula analítica exacta que tiene en cuenta la correlación entre los dos clasificadores. Eso es justo lo que se necesita para responder qué modelo ordena mejor los casos, pero no dice nada sobre qué pasaría si alguno de estos modelos se usara de verdad para aprobar o rechazar solicitudes con un punto de corte fijo. Esta sección responde una pregunta distinta y complementaria: a un umbral de decisión real, ¿un modelo se equivoca con más frecuencia que el otro, y en qué dirección?

Para eso usamos dos herramientas. La prueba de McNemar compara las tasas de acierto de dos clasificadores en los mismos casos, a través de una tabla de concordancia 2x2 que muestra en qué casos acierta uno y falla el otro, y viceversa. El bootstrap pareado, por su parte, da un intervalo de confianza no paramétrico para la diferencia de AUC, AUC-PR y F1, que no depende de la aproximación asintótica normal que usa DeLong. Esto sirve como chequeo independiente, y además es necesario para dos métricas (AUC-PR y F1) que no tienen una fórmula de varianza tan simple como la de DeLong.

Algo importante: cualquier umbral de clasificación que usamos en esta sección se elige solo con datos de entrenamiento. Elegirlo mirando el conjunto de prueba (por ejemplo buscando el umbral que maximiza el F1 directamente en el test) filtraría información del test hacia una decisión que después se evalúa en ese mismo test, un sesgo optimista clásico. Con el umbral fijado de antemano en entrenamiento, la evaluación en test queda honesta.

In [1]:
import sys
sys.path.insert(0, "../src")
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mcnemar_bootstrap import umbral_youden, mcnemar_test, bootstrap_pareado
from delong import holm_correction

pd.set_option("display.width", 160)
print("Modulo mcnemar_bootstrap.py cargado correctamente.")


Modulo mcnemar_bootstrap.py cargado correctamente.


## 6.1. Umbral de clasificación: índice de Youden, calculado solo con datos de entrenamiento

El índice J de Youden se define como `J = sensibilidad + especificidad - 1`, que también se puede escribir como la tasa de verdaderos positivos menos la tasa de falsos positivos, evaluado en cada punto de la curva ROC. El umbral que maximiza J es el que mejor balancea los dos tipos de error (falsos positivos y falsos negativos), a diferencia del umbral por defecto de 0,5 (o de margen 0 para los modelos lineales), que no tiene ninguna razón particular para ser el óptimo en un problema tan desbalanceado como este, con apenas 12% de incumplimiento.

```python
def umbral_youden(y_train, score_train):\n    fpr, tpr, thresholds = roc_curve(y_train, score_train)\n    j = tpr - fpr\n    return float(thresholds[np.argmax(j)])
```

Calculamos este umbral una vez por modelo y por entorno, usando únicamente `y_train` y `score_train`. En ningún momento de este paso tocamos el conjunto de prueba.

In [2]:
sk_train = pd.read_parquet("../data/sklearn_train_scores.parquet")
sk_test = pd.read_parquet("../data/sklearn_test_scores.parquet")
sp_train = pd.read_parquet("../data/spark_train_scores_raw.parquet")
sp_test = pd.read_parquet("../data/spark_test_scores.parquet")
for df in (sk_train, sk_test, sp_train, sp_test):
    df["id"] = df["id"].astype(str)

# Alineacion por id (igual que en la seccion 5): todo lo que compare ambos entornos se lee de aqui.
test_merged = sk_test.merge(sp_test, on="id", suffixes=("_sk", "_sp"))
assert (test_merged["default_sk"] == test_merged["default_sp"]).all()
y_test = test_merged["default_sk"].values
print(f"Test alineado: {len(test_merged):,} filas (tasa de default: {y_test.mean()*100:.2f}%)")


Test alineado: 452,141 filas (tasa de default: 11.88%)


La mayoría de los umbrales resultaron razonables y parecidos entre los dos entornos (regresión logística, árbol, bosque y GBT quedaron todos entre 0,12 y 0,13, coherente con que alrededor del 12% de la base es la clase positiva). Dos casos se salen de ese patrón y vale la pena mirarlos con más cuidado.

In [3]:
u_demo = umbral_youden(sk_train["default"].values, sk_train["score_LogisticRegression"].values)
print(f"Umbral Youden (recalculado ahora): {u_demo:.6f}")


Umbral Youden (recalculado ahora): 0.121438


In [4]:
import json
umbrales = json.load(open("../outputs/tables/umbrales_youden.json"))
tabla_umbrales = pd.DataFrame([
    {"modelo": k.rsplit("_", 1)[0], "entorno": k.rsplit("_", 1)[1], "umbral": v}
    for k, v in umbrales.items()
]).pivot(index="modelo", columns="entorno", values="umbral")
tabla_umbrales


entorno,sklearn,spark
modelo,,
DecisionTree,0.119114,0.118904
GBT,0.122532,0.121548
LinearSVC,-1.000000,-1.000062
LogisticRegression,0.121438,0.121069
NaiveBayes,1.000000,0.000273
RandomForest,0.127844,0.127766


### 6.1.1. Los dos umbrales atípicos: por qué están ahí

In [5]:
for nombre, col in [("sklearn LinearSVC", ("score_LinearSVC", sk_train)),
                    ("spark LinearSVC",   ("score_LinearSVC", sp_train)),
                    ("sklearn GaussianNB (NaiveBayes)", ("score_GaussianNB", sk_train)),
                    ("spark NaiveBayes",  ("score_NaiveBayes", sp_train))]:
    c, df = col
    s = df[c].values
    print(f"{nombre:34s} min={s.min():9.4f}  max={s.max():9.4f}  mean={s.mean():8.4f}  std={s.std():.4f}")


sklearn LinearSVC                  min=  -1.0000  max=  -1.0000  mean= -1.0000  std=0.0000
spark LinearSVC                    min=  -1.3429  max=  -0.9974  mean= -1.0261  std=0.0588
sklearn GaussianNB (NaiveBayes)    min=   0.0000  max=   1.0000  mean=  0.8340  std=0.3642
spark NaiveBayes                   min=   0.0000  max=   1.0000  mean=  0.1851  std=0.3659


`LinearSVC` en scikit-learn quedó con una desviación de 0 y un valor mínimo y máximo exactamente iguales a -1. Esto no es un umbral elegido entre varias alternativas, es el síntoma, en los datos de entrenamiento, del mismo colapso que diagnosticamos en la sección 3.7: el modelo le asigna literalmente el mismo puntaje a las 1.808.560 filas de entrenamiento. Con un puntaje constante no existe ninguna curva ROC real, porque cualquier punto de corte por encima o por debajo de -1 da la misma predicción para todo el mundo, así que el valor que devuelve la función sobre esa meseta plana es básicamente arbitrario. Es un umbral degenerado, y así lo reportamos en las conclusiones.

`LinearSVC` en Spark, en cambio, no está colapsado: el puntaje sí varía (desviación de 0,059), pero está todo concentrado en valores negativos, entre -1,34 y -1,00. Con el umbral natural de margen 0, este modelo también predeciría \"no default\" para el 100% de los casos, igual que scikit-learn a ese umbral. Pero el índice de Youden encuentra, dentro de ese rango tan angosto, el punto donde la separación entre clases es mejor (alrededor de -1,0001), y ese punto sí recupera capacidad predictiva real, de ahí el AUC de 0,704 que vimos en la sección 4. Es un umbral extremo, pero no degenerado, y es la prueba de que elegir el punto de corte con datos de entrenamiento, en vez de usar 0 por defecto, sí importa.

`GaussianNB` en scikit-learn tuvo una media de 0,83, con 369.821 de las 1.808.560 filas (casi 20%) exactamente en 1,0. La probabilidad se satura en el techo para una fracción enorme de los datos, lo cual es otro reflejo de la mala calibración que ya habíamos visto en la sección 3.6 (recall de 0,96 y precisión de 0,24, es decir que el modelo básicamente dice \"default\" casi siempre). El umbral óptimo termina justo en el borde de esa saturación.

`NaiveBayes` en Spark está mucho menos saturado (solo alrededor de 3% de las filas en el máximo), con una distribución de probabilidad más extendida, lo cual es consistente con su AUC de 0,699, apenas por encima del de scikit-learn (0,694, ya corregido en la sección 4.6c).

En resumen, de los cuatro casos atípicos, solo uno (`LinearSVC` en scikit-learn) es realmente degenerado. Los otros tres son umbrales extremos pero genuinos, que reflejan qué tan bien o mal calibrado está cada modelo.

## 6.2. Las 8 comparaciones: 6 entre entornos más el par de los dos mejores AUC dentro de cada entorno

In [6]:
res_sk = pd.read_csv("../outputs/tables/sklearn_results.csv").set_index("modelo")
res_sp = pd.read_csv("../outputs/tables/spark_results.csv").set_index("modelo")
top2_sk = res_sk["test_auc"].sort_values(ascending=False).index[:2].tolist()
top2_sp = res_sp["test_auc"].sort_values(ascending=False).index[:2].tolist()
print("Top-2 AUC en scikit-learn:", top2_sk)
print("Top-2 AUC en PySpark     :", top2_sp)
print()
print("Las 8 comparaciones: 6 pares entre entornos (misma familia de modelo, scikit-learn vs. PySpark)")
print("                    + top2_dentro_sklearn (los 2 mejores AUC de scikit-learn entre si)")
print("                    + top2_dentro_spark    (los 2 mejores AUC de PySpark entre si)")


Top-2 AUC en scikit-learn: ['GBT_HistGB', 'DecisionTree']
Top-2 AUC en PySpark     : ['GBT', 'LogisticRegression']

Las 8 comparaciones: 6 pares entre entornos (misma familia de modelo, scikit-learn vs. PySpark)
                    + top2_dentro_sklearn (los 2 mejores AUC de scikit-learn entre si)
                    + top2_dentro_spark    (los 2 mejores AUC de PySpark entre si)


Aplicamos la corrección de Holm dentro de este único grupo de 8 comparaciones, sin mezclarlo con las 36 de la sección 5, porque son preguntas distintas: aquí nos interesa controlar la tasa de falsos descubrimientos entre las 8 decisiones de McNemar que vamos a tomar, no combinarlas con las comparaciones de AUC de la sección anterior.

## 6.3. Prueba de McNemar: tabla de concordancia más corrección de Holm

Para cada par de modelos, con las predicciones binarias ya con el umbral de Youden correspondiente aplicado, construimos una tabla de concordancia de aciertos, con cuatro casillas: ambos aciertan, solo A acierta, solo B acierta, y ninguno acierta. McNemar prueba si el número de casos donde solo A acierta es igual al número de casos donde solo B acierta, es decir, si ningún modelo se equivoca sistemáticamente más que el otro en los casos donde justo difieren. Usamos la versión exacta (binomial) cuando la suma de esos dos números discordantes es menor a 25, y la versión asintótica con corrección de continuidad en caso contrario, el mismo criterio que usa `statsmodels.stats.contingency_tables.mcnemar`.

```python
def mcnemar_test(y_true, pred_a, pred_b):\n    correct_a, correct_b = (pred_a == y_true), (pred_b == y_true)\n    tabla = np.array([[ (correct_a & correct_b).sum(),  (correct_a & ~correct_b).sum() ],\n                       [ (~correct_a & correct_b).sum(), (~correct_a & ~correct_b).sum() ]])\n    b, c = tabla[0,1], tabla[1,0]\n    return mcnemar(tabla, exact=(b+c < 25), correction=True)
```

Calculamos McNemar siempre sobre las 452.141 filas completas del test, a diferencia del bootstrap de la sección 6.4, que usa una submuestra por costo computacional. Es una prueba barata, así que no hace falta reducir nada acá.

Con hasta 452.141 pares de observaciones, hasta una diferencia modesta en la tasa de error se vuelve estadísticamente detectable, y las 8 comparaciones rechazan la hipótesis nula después de Holm, igual que pasó con las 36 comparaciones de DeLong en la sección 5. El caso más extremo es `LinearSVC` entre entornos, con 193.719 pares discordantes (un 43% de los casos donde los dos modelos difieren), lo cual tiene sentido porque uno de los dos está colapsado prediciendo \"no default\" casi siempre y el otro no. El caso con menor desacuerdo es `NaiveBayes` entre entornos, con apenas 78 pares discordantes, pero esto es un poco engañoso: los dos umbrales (1,0 y 0,0003) son tan extremos que casi ningún caso se clasifica como \"default\" en ninguno de los dos entornos. Hay poco desacuerdo porque hay muy pocas predicciones positivas en general, no porque los modelos sean parecidos.

In [7]:
mcnemar_resultados = pd.read_csv("../outputs/tables/mcnemar_resultados.csv")
cols_mc = ["par","tipo","n10_A_acierta_B_falla","n01_A_falla_B_acierta","b_mas_c","exacta",
           "statistic","p_value","p_holm","rechaza_h0_holm_0.05"]
mcnemar_resultados[cols_mc].round(4)


,par,tipo,n10_A_acierta_B_falla,n01_A_falla_B_acierta,b_mas_c,exacta,statistic,p_value,p_holm,rechaza_h0_holm_0.05
0,LogisticRegression,entre_entornos,2101,3231,5332,False,239.0550,0.0000,0.0000,True
1,DecisionTree,entre_entornos,28676,29796,58472,False,21.4147,0.0000,0.0000,True
2,RandomForest,entre_entornos,9849,7691,17540,False,265.2594,0.0000,0.0000,True
3,GBT,entre_entornos,12359,11947,24306,False,6.9498,0.0084,0.0084,True
4,LinearSVC,entre_entornos,77932,115787,193719,False,7396.9271,0.0000,0.0000,True
5,NaiveBayes,entre_entornos,14,64,78,False,30.7821,0.0000,0.0000,True
6,GBT_HistGB_vs_DecisionTree,top2_dentro_sklearn,30165,35431,65596,False,422.5902,0.0000,0.0000,True
7,GBT_vs_LogisticRegression,top2_dentro_spark,24609,20443,45052,False,385.0489,0.0000,0.0000,True


## 6.4. Bootstrap pareado: diferencias de AUC, AUC-PR y F1 con intervalo de confianza del 95%

El bootstrap pareado remuestrea los mismos índices para los dos modelos en cada iteración, lo que preserva la correlación entre A y B, igual que hace DeLong de forma analítica, y con eso construye un intervalo de confianza percentil del 95% para la diferencia observada en tres métricas: AUC, AUC-PR (más informativa que el AUC bajo desbalance de clases) y F1, que a diferencia de las otras dos sí depende del umbral de Youden que elegimos en la sección 6.1.

```python
def bootstrap_pareado(y_true, score_a, score_b, pred_a, pred_b, B=2000, seed=42):\n    rng = np.random.default_rng(seed)\n    deltas_auc, deltas_auc_pr, deltas_f1 = [], [], []\n    for _ in range(B):\n        idx = rng.integers(0, len(y_true), size=len(y_true))\n        yb = y_true[idx]\n        if yb.sum() in (0, len(yb)):\n            continue  # remuestreo degenerado, se descarta\n        deltas_auc.append(roc_auc_score(yb, score_a[idx]) - roc_auc_score(yb, score_b[idx]))\n        # (analogo para AUC-PR y F1)\n    ...  # IC95% percentil de cada lista de deltas
```

### Por qué usamos una submuestra de 50.000 filas y no las 452.141 completas

Al medir el tiempo de una sola iteración de bootstrap sobre las 452.141 filas completas, nos dio 0,73 segundos por iteración, lo que con B=2000 y 8 comparaciones significaría cerca de 194 minutos solo para este chequeo. Esto es poco práctico para lo que es, por diseño, un chequeo complementario a DeLong, que sí corrió sobre el test completo en la sección 5. Con una submuestra aleatoria estratificada (semilla fija 42, preservando la tasa de incumplimiento) de 50.000 filas, el costo por iteración baja a 0,084 segundos, es decir unos 22 minutos en total, sin comprometer la validez del intervalo de confianza, porque 50.000 sigue siendo una muestra grande para un bootstrap. McNemar, en cambio, sí usa las 452.141 filas completas, porque ahí el costo es insignificante.

En las 8 comparaciones, el intervalo de la diferencia de AUC excluye el cero, coherente otra vez con McNemar y con DeLong. La comparación más cercana a cero es `LogisticRegression` entre entornos (diferencia de -0,0011, intervalo de -0,0014 a -0,0007): un intervalo angosto, alejado de cero, pero de una magnitud diminuta. Es el ejemplo más claro, en esta sección, de algo estadísticamente significativo pero prácticamente irrelevante.

In [8]:
bootstrap_resultados = pd.read_csv("../outputs/tables/bootstrap_resultados.csv")
cols_bt = ["par","tipo","delta_auc","auc_ci_low","auc_ci_high","auc_excluye_cero",
           "delta_auc_pr","auc_pr_ci_low","auc_pr_ci_high","delta_f1","f1_ci_low","f1_ci_high"]
bootstrap_resultados[cols_bt].round(4)


,par,tipo,delta_auc,auc_ci_low,auc_ci_high,auc_excluye_cero,delta_auc_pr,auc_pr_ci_low,auc_pr_ci_high,delta_f1,f1_ci_low,f1_ci_high
0,LogisticRegression,entre_entornos,-0.0011,-0.0014,-0.0007,True,-0.0003,-0.0010,0.0005,-0.0010,-0.0023,0.0003
1,DecisionTree,entre_entornos,0.0144,0.0100,0.0186,True,0.0113,0.0061,0.0164,0.0063,0.0019,0.0107
2,RandomForest,entre_entornos,0.0044,0.0035,0.0054,True,0.0040,0.0022,0.0055,0.0048,0.0026,0.0070
3,GBT,entre_entornos,0.0060,0.0046,0.0073,True,0.0060,0.0036,0.0086,0.0036,0.0011,0.0062
4,LinearSVC,entre_entornos,-0.1371,-0.1470,-0.1273,True,-0.0896,-0.0978,-0.0816,-0.0881,-0.0959,-0.0806
5,NaiveBayes,entre_entornos,-0.0055,-0.0065,-0.0046,True,-0.0209,-0.0255,-0.0167,0.0001,-0.0001,0.0004
6,GBT_HistGB_vs_DecisionTree,top2_dentro_sklearn,0.0049,0.0012,0.0084,True,0.0228,0.0167,0.0291,-0.0002,-0.0046,0.0043
7,GBT_vs_LogisticRegression,top2_dentro_spark,0.0105,0.0079,0.0130,True,0.0092,0.0041,0.0138,0.0091,0.0056,0.0128


## 6.5. Tabla resumen final: ¿coinciden DeLong, McNemar y el bootstrap?

El enunciado pide comparar las conclusiones de las tres pruebas en una sola tabla. Las cruzamos por nombre de par, traduciendo cuando hace falta entre el nombre de familia que usa DeLong y el nombre de resultados que usan McNemar y el bootstrap (por ejemplo `GBT_HistGB` se traduce a `GBT`).

Las tres pruebas están de acuerdo en la dirección (el signo de la diferencia de AUC es el mismo en DeLong y en el bootstrap en las 8 comparaciones) y en la significancia estadística (las 8 rechazan la hipótesis nula en las tres pruebas). Con una n tan grande esto era predecible, como ya discutimos en la sección 5, pero igual es una buena señal de consistencia interna: tres formas distintas de medir si dos modelos son diferentes (ranking por covarianza analítica, aciertos a un umbral fijo, y remuestreo empírico) llegan a la misma respuesta.

In [9]:
nombre_a_familia = {"LogisticRegression": "LogisticRegression", "DecisionTree": "DecisionTree",
                     "RandomForest": "RandomForest", "GBT_HistGB": "GBT", "GBT": "GBT",
                     "LinearSVC": "LinearSVC", "GaussianNB": "NaiveBayes", "NaiveBayes": "NaiveBayes"}

delong_between = pd.read_csv("../outputs/tables/delong_between_envs.csv")
delong_within_sk = pd.read_csv("../outputs/tables/delong_within_sklearn.csv")
delong_within_sp = pd.read_csv("../outputs/tables/delong_within_spark.csv")


def familia_de_resultado(etiqueta):
    return nombre_a_familia[re.sub(r'_(sklearn|spark)$', '', etiqueta)]


def delong_para(tipo, par, modelo_a_mc, modelo_b_mc):
    """Devuelve (delta_auc, p_holm, rechaza) de DeLong, orientado igual que McNemar/bootstrap
    (modelo_a - modelo_b), buscando la fila correspondiente en la tabla de DeLong que aplique."""
    if tipo == "entre_entornos":
        f = delong_between[delong_between["modelo_a"] == f"sk_{par}"].iloc[0]
        return f["delta_auc"], f["p_holm"], bool(f["rechaza_h0_holm_0.05"])
    tabla = delong_within_sk if tipo == "top2_dentro_sklearn" else delong_within_sp
    na, nb_ = familia_de_resultado(modelo_a_mc), familia_de_resultado(modelo_b_mc)
    pre = "sk_" if tipo == "top2_dentro_sklearn" else "sp_"
    fila = tabla[(tabla.modelo_a == pre+na) & (tabla.modelo_b == pre+nb_)]
    if len(fila):
        f = fila.iloc[0]
        return f["delta_auc"], f["p_holm"], bool(f["rechaza_h0_holm_0.05"])
    f = tabla[(tabla.modelo_a == pre+nb_) & (tabla.modelo_b == pre+na)].iloc[0]
    return -f["delta_auc"], f["p_holm"], bool(f["rechaza_h0_holm_0.05"])


filas = []
for _, m in mcnemar_resultados.iterrows():
    b = bootstrap_resultados[(bootstrap_resultados.tipo == m.tipo) & (bootstrap_resultados.par == m.par)].iloc[0]
    dd, dp, drej = delong_para(m.tipo, m.par, m.modelo_a, m.modelo_b)
    filas.append({
        "par": m.par, "tipo": m.tipo,
        "DeLong_ΔAUC": dd, "DeLong_rechaza_H0": drej, "DeLong_práctico(≥0.005)": abs(dd) >= 0.005,
        "McNemar_rechaza_H0": bool(m["rechaza_h0_holm_0.05"]),
        "Bootstrap_ΔAUC": b["delta_auc"], "Bootstrap_IC95%": f"[{b['auc_ci_low']:.4f}, {b['auc_ci_high']:.4f}]",
        "Bootstrap_excluye_0": bool(b["auc_excluye_cero"]),
    })
tabla_resumen = pd.DataFrame(filas)
tabla_resumen["las_3_coinciden_en_signo_y_significancia"] = (
    tabla_resumen["DeLong_rechaza_H0"] & tabla_resumen["McNemar_rechaza_H0"] & tabla_resumen["Bootstrap_excluye_0"]
)
tabla_resumen.round(4)


,par,tipo,DeLong_ΔAUC,DeLong_rechaza_H0,DeLong_práctico(≥0.005),McNemar_rechaza_H0,Bootstrap_ΔAUC,Bootstrap_IC95%,Bootstrap_excluye_0,las_3_coinciden_en_signo_y_significancia
0,LogisticRegression,entre_entornos,-0.0011,True,False,True,-0.0011,"[-0.0014, -0.0007]",True,True
1,DecisionTree,entre_entornos,0.0164,True,True,True,0.0144,"[0.0100, 0.0186]",True,True
2,RandomForest,entre_entornos,0.0039,True,False,True,0.0044,"[0.0035, 0.0054]",True,True
3,GBT,entre_entornos,0.0066,True,True,True,0.0060,"[0.0046, 0.0073]",True,True
4,LinearSVC,entre_entornos,-0.1443,True,True,True,-0.1371,"[-0.1470, -0.1273]",True,True
5,NaiveBayes,entre_entornos,-0.0055,True,True,True,-0.0055,"[-0.0065, -0.0046]",True,True
6,GBT_HistGB_vs_DecisionTree,top2_dentro_sklearn,0.0025,True,False,True,0.0049,"[0.0012, 0.0084]",True,True
7,GBT_vs_LogisticRegression,top2_dentro_spark,0.0095,True,True,True,0.0105,"[0.0079, 0.0130]",True,True


Donde sí aparece un matiz es en la significancia práctica, usando el umbral de 0,005 que ya habíamos declarado en la sección 5. `RandomForest` entre entornos (DeLong da una diferencia de 0,0039, por debajo del umbral) y `GBT_HistGB` contra `DecisionTree` dentro de scikit-learn (DeLong da 0,0025) son los dos casos más ajustados. El bootstrap los ubica un poco más alto (0,0044 y 0,0049 respectivamente), pero igual por debajo o justo en el límite de 0,005. La conclusión práctica es la misma con las dos pruebas para ambos casos: una diferencia real pero pequeña, del tamaño en el que \"estadísticamente significativo\" e \"importa para elegir un modelo en la práctica\" empiezan a separarse.

## 6.6. Limitaciones de McNemar y del bootstrap pareado en este caso

- McNemar depende por completo del umbral que se elija. Un punto de corte distinto, igualmente válido, puede cambiar cuántos casos son discordantes y en qué dirección, a diferencia de DeLong, que compara el ranking completo sin fijar ningún umbral. Por eso esta prueba es complementaria y no un sustituto: responde a qué modelo se equivoca más en este punto de operación concreto, no a cuál ordena mejor en general.
- McNemar no mide magnitud, solo la dirección de los aciertos discordantes. Con una n tan grande, cualquier asimetría, sin importar cuán chica, se vuelve significativa, la misma advertencia de significancia estadística contra significancia práctica de la sección 5, pero aquí sin un equivalente tan claro al tamaño de efecto como la diferencia de AUC.
- El bootstrap de esta sección corrió sobre una submuestra fija de 50.000 filas, no las 452.141 completas, por la decisión de costo computacional que explicamos en la sección 6.4. Esto es distinto al resto del proyecto, que evita cualquier reducción de escala en los datos de entrenamiento y de prueba. Queda razonablemente justificado porque es un chequeo complementario a DeLong, que sí corrió a escala completa, y porque 50.000 sigue siendo una muestra grande para un intervalo de bootstrap, pero con más tiempo de cómputo disponible, correr las 452.141 filas completas sería la versión más rigurosa.
- Las dos pruebas, igual que DeLong, evalúan un solo split de entrenamiento y prueba. Ninguna de las tres mide la variabilidad que vendría de repetir todo el proceso (la partición, el ajuste de hiperparámetros, la elección del umbral) con una semilla distinta. Esa sería una pregunta de validación cruzada anidada, que queda fuera del alcance de esta comparación puntual.